# 📚 Citations in RAG Systems

**Provide source attribution for trustworthy AI answers**

---

## 📋 Overview

**What you'll learn:**
- Why citations matter
- Extracting and formatting citations
- Tracking source attribution
- Citation verification
- Production citation systems

**Time estimate:** ⏱️ 40 minutes | **Difficulty:** 🟡 Intermediate

---

In [ ]:
from openai import OpenAI
from typing import List, Dict
import re
import os

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

print("✅ Setup complete")

## 🤔 Why Citations Matter

### Problems without Citations:
```
User: "What is our refund policy?"
AI: "We offer 30-day refunds"
User: "Where did you get that?" 🤷
```

### With Citations:
```
User: "What is our refund policy?"
AI: "We offer 30-day refunds [1]"

Sources:
[1] Refund Policy - docs/policies.md
```

### Benefits:
- ✅ **Trust**: Users can verify
- ✅ **Transparency**: See what AI read
- ✅ **Debugging**: Find wrong sources
- ✅ **Legal**: Track information sources
- ✅ **Compliance**: Required in some domains

## 📝 Basic Citation System

In [ ]:
# Sample documents with metadata
documents = [
    {
        "id": "doc1",
        "text": "Our company offers a 30-day money-back guarantee on all products.",
        "source": "Refund Policy",
        "url": "https://example.com/refund-policy",
        "updated": "2024-01-15"
    },
    {
        "id": "doc2",
        "text": "Shipping is free for orders over $50 within the continental US.",
        "source": "Shipping Policy",
        "url": "https://example.com/shipping",
        "updated": "2024-01-10"
    },
    {
        "id": "doc3",
        "text": "Customer support is available 24/7 via email and live chat.",
        "source": "Support Information",
        "url": "https://example.com/support",
        "updated": "2024-01-20"
    },
]

class RAGWithCitations:
    """RAG system that provides citations."""
    
    def __init__(self, documents: List[Dict]):
        self.documents = {doc['id']: doc for doc in documents}
    
    def generate_answer(
        self,
        question: str,
        retrieved_doc_ids: List[str]
    ) -> Dict:
        """Generate answer with citations."""
        
        # Build context with citation markers
        context_parts = []
        for i, doc_id in enumerate(retrieved_doc_ids, 1):
            doc = self.documents[doc_id]
            context_parts.append(f"[{i}] {doc['text']}")
        
        context = "\n".join(context_parts)
        
        # Create prompt with citation instructions
        prompt = f"""Answer the question using the provided context.
Include citations using [1], [2], etc. after each fact.

Context:
{context}

Question: {question}

Answer (with citations):"""
        
        # Get LLM response
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=200
        )
        
        answer_text = response.choices[0].message.content
        
        # Extract citation numbers from answer
        cited_nums = self._extract_citations(answer_text)
        
        # Build citation list
        citations = []
        for i, doc_id in enumerate(retrieved_doc_ids, 1):
            if i in cited_nums:
                doc = self.documents[doc_id]
                citations.append({
                    'number': i,
                    'source': doc['source'],
                    'url': doc['url'],
                    'text': doc['text'],
                    'updated': doc['updated']
                })
        
        return {
            'answer': answer_text,
            'citations': citations,
            'context_used': context
        }
    
    def _extract_citations(self, text: str) -> set:
        """Extract citation numbers like [1], [2] from text."""
        # Find all [N] patterns
        matches = re.findall(r'\[(\d+)\]', text)
        return set(int(m) for m in matches)
    
    def format_response(self, result: Dict) -> str:
        """Format answer with citations for display."""
        output = []
        output.append(result['answer'])
        
        if result['citations']:
            output.append("\n\nSources:")
            for cite in result['citations']:
                output.append(f"[{cite['number']}] {cite['source']}")
                output.append(f"    {cite['url']}")
                output.append(f"    Updated: {cite['updated']}")
        
        return "\n".join(output)

# Test
rag = RAGWithCitations(documents)

question = "What is your refund policy?"
retrieved_ids = ["doc1"]  # Simulated retrieval

result = rag.generate_answer(question, retrieved_ids)

print("📚 RAG with Citations\n")
print(rag.format_response(result))

## 🎯 Inline vs Footnote Citations

In [ ]:
class CitationFormatter:
    """Different citation styles."""
    
    @staticmethod
    def inline_style(answer: str, citations: List[Dict]) -> str:
        """Inline citations like [1]."""
        output = [answer, "\n\nSources:"]
        for cite in citations:
            output.append(f"[{cite['number']}] {cite['source']} - {cite['url']}")
        return "\n".join(output)
    
    @staticmethod
    def parenthetical_style(answer: str, citations: List[Dict]) -> str:
        """Parenthetical citations like (Source, 2024)."""
        # Replace [1] with (Source, Date)
        for cite in citations:
            year = cite['updated'][:4]
            pattern = f"\[{cite['number']}\]"
            replacement = f"({cite['source']}, {year})"
            answer = re.sub(pattern, replacement, answer)
        return answer
    
    @staticmethod
    def linked_style(answer: str, citations: List[Dict]) -> str:
        """Markdown-style links."""
        # Replace [1] with [Source](url)
        for cite in citations:
            pattern = f"\[{cite['number']}\]"
            replacement = f"[{cite['source']}]({cite['url']})"
            answer = re.sub(pattern, replacement, answer)
        return answer

# Test different styles
formatter = CitationFormatter()

print("🎨 Citation Styles\n")
print("="*60)

print("\n1. Inline Style:")
print(formatter.inline_style(result['answer'], result['citations']))

print("\n2. Parenthetical Style:")
print(formatter.parenthetical_style(result['answer'], result['citations']))

print("\n3. Linked Style (Markdown):")
print(formatter.linked_style(result['answer'], result['citations']))

## ✅ Citation Verification

In [ ]:
class CitationVerifier:
    """Verify that citations support the answer."""
    
    @staticmethod
    def verify(
        answer: str,
        citations: List[Dict],
        question: str
    ) -> Dict:
        """Verify citations are relevant and used correctly."""
        
        issues = []
        
        # Check 1: Are citations present?
        cited_nums = set(re.findall(r'\[(\d+)\]', answer))
        if not cited_nums:
            issues.append("No citations found in answer")
        
        # Check 2: Are all citations valid?
        valid_nums = {c['number'] for c in citations}
        invalid = cited_nums - valid_nums
        if invalid:
            issues.append(f"Invalid citation numbers: {invalid}")
        
        # Check 3: Are citations used?
        unused = valid_nums - cited_nums
        if unused:
            issues.append(f"Provided but unused citations: {unused}")
        
        # Check 4: Do citations support the answer? (LLM-based)
        support_check = CitationVerifier._check_support(
            answer,
            citations,
            question
        )
        
        return {
            'is_valid': len(issues) == 0,
            'issues': issues,
            'support_check': support_check
        }
    
    @staticmethod
    def _check_support(answer: str, citations: List[Dict], question: str) -> Dict:
        """Use LLM to verify citations support the answer."""
        
        # Build sources text
        sources_text = "\n".join([
            f"[{c['number']}] {c['text']}"
            for c in citations
        ])
        
        prompt = f"""Check if the answer is supported by the sources.

Question: {question}

Answer: {answer}

Sources:
{sources_text}

Is the answer fully supported by the sources? Respond with:
- "SUPPORTED" if all claims have citations
- "PARTIALLY" if some claims lack support
- "NOT SUPPORTED" if answer contradicts sources

Response:"""
        
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=50
        )
        
        verdict = response.choices[0].message.content.strip()
        
        return {
            'verdict': verdict,
            'is_supported': 'SUPPORTED' in verdict.upper()
        }

# Test verification
verifier = CitationVerifier()

verification = verifier.verify(
    result['answer'],
    result['citations'],
    question
)

print("✅ Citation Verification\n")
print(f"Valid: {verification['is_valid']}")
print(f"Support: {verification['support_check']['verdict']}")
if verification['issues']:
    print(f"\nIssues:")
    for issue in verification['issues']:
        print(f"  - {issue}")

## 🏗️ Production Citation System

In [ ]:
from dataclasses import dataclass
from datetime import datetime

@dataclass
class Citation:
    """Citation with full metadata."""
    number: int
    doc_id: str
    source_name: str
    url: str
    excerpt: str
    updated_at: str
    confidence: float = 1.0

class ProductionRAGCitations:
    """Production-ready RAG with comprehensive citations."""
    
    def __init__(self, documents: List[Dict]):
        self.documents = {doc['id']: doc for doc in documents}
    
    def generate_with_citations(
        self,
        question: str,
        retrieved_docs: List[Dict],
        citation_style: str = 'inline'
    ) -> Dict:
        """Generate answer with citations and verification."""
        
        # Build context
        context = self._build_context(retrieved_docs)
        
        # Generate answer
        answer = self._generate_answer(question, context)
        
        # Extract and format citations
        citations = self._extract_citations(answer, retrieved_docs)
        
        # Verify
        verification = self._verify_citations(answer, citations, question)
        
        # Format
        formatted = self._format_output(answer, citations, citation_style)
        
        return {
            'answer': answer,
            'formatted_answer': formatted,
            'citations': [vars(c) for c in citations],
            'verification': verification,
            'metadata': {
                'question': question,
                'num_sources': len(retrieved_docs),
                'num_citations': len(citations),
                'timestamp': datetime.now().isoformat()
            }
        }
    
    def _build_context(self, docs: List[Dict]) -> str:
        """Build numbered context."""
        return "\n".join([
            f"[{i+1}] {doc['text']}"
            for i, doc in enumerate(docs)
        ])
    
    def _generate_answer(self, question: str, context: str) -> str:
        """Generate answer with citations."""
        prompt = f"""Answer using the context. Cite sources with [1], [2], etc.

Context:
{context}

Question: {question}

Answer:"""
        
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        
        return response.choices[0].message.content
    
    def _extract_citations(self, answer: str, docs: List[Dict]) -> List[Citation]:
        """Extract citations from answer."""
        cited_nums = set(int(m) for m in re.findall(r'\[(\d+)\]', answer))
        
        citations = []
        for i, doc in enumerate(docs, 1):
            if i in cited_nums:
                citations.append(Citation(
                    number=i,
                    doc_id=doc['id'],
                    source_name=doc['source'],
                    url=doc['url'],
                    excerpt=doc['text'][:100],
                    updated_at=doc['updated']
                ))
        
        return citations
    
    def _verify_citations(self, answer: str, citations: List[Citation], question: str) -> Dict:
        """Basic verification."""
        return {
            'has_citations': len(citations) > 0,
            'num_citations': len(citations)
        }
    
    def _format_output(self, answer: str, citations: List[Citation], style: str) -> str:
        """Format with citations."""
        output = [answer, "\n\nSources:"]
        for cite in citations:
            output.append(f"[{cite.number}] {cite.source_name}")
            output.append(f"    {cite.url}")
            output.append(f"    Updated: {cite.updated_at}")
        return "\n".join(output)

# Test production system
prod_rag = ProductionRAGCitations(documents)

result = prod_rag.generate_with_citations(
    question="What are your shipping and refund policies?",
    retrieved_docs=[documents[0], documents[1]]
)

print("🏗️ Production RAG with Citations\n")
print(result['formatted_answer'])
print(f"\nMetadata: {result['metadata']}")

## ✅ Summary

### Citation Best Practices:

1. **Always provide sources**
   ```python
   # ✅ Good
   "We offer 30-day refunds [1]"
   
   # ❌ Bad
   "We offer 30-day refunds"  # Where from?
   ```

2. **Include metadata**
   ```python
   citation = {
       'source': 'Refund Policy',
       'url': 'https://...',
       'updated': '2024-01-15',  # Freshness
       'excerpt': '...'           # Verification
   }
   ```

3. **Verify citations**
   - Check if citation numbers are valid
   - Verify sources support claims
   - Flag unsupported statements

4. **Format for readability**
   - Inline: [1], [2]
   - Parenthetical: (Source, 2024)
   - Linked: [Source](url)

### Citation Styles:

**Inline (Academic):**
```
Answer text [1] more text [2].

Sources:
[1] Source Name - URL
[2] Another Source - URL
```

**Parenthetical:**
```
Answer text (Source, 2024) more text.
```

**Linked:**
```markdown
Answer text [Source](url) more text.
```

### Prompting for Citations:

```python
prompt = f"""
Answer the question using ONLY the provided context.
Cite every fact with [1], [2], etc.
If information is not in context, say "I don't have that information."

Context:
[1] {source1}
[2] {source2}

Question: {question}
Answer with citations:
"""
```

### Verification Checklist:

- ✅ Answer has citations?
- ✅ All citations are valid?
- ✅ Citations support claims?
- ✅ No unsupported claims?
- ✅ Sources are recent?

### Production Tips:

1. **Log citations**
   ```python
   # Track which sources are used
   logger.info(f"Used sources: {citation_ids}")
   ```

2. **Make clickable**
   ```python
   # Include URLs for easy verification
   citation['url'] = full_url
   ```

3. **Show excerpts**
   ```python
   # Let users see the source text
   citation['excerpt'] = source_text[:200]
   ```

4. **Track freshness**
   ```python
   # Show when source was updated
   citation['updated_at'] = doc.updated
   ```

### Common Issues:

**Issue 1: No citations**
- Solution: Explicit prompting
- Example: "You MUST cite sources"

**Issue 2: Invalid citations**
- Solution: Verify before showing
- Example: Check citation numbers exist

**Issue 3: Hallucinated sources**
- Solution: Only allow provided sources
- Example: Constrain to [1], [2], [3]

### Next: `05_rag_systems/08_advanced_chunking.ipynb`